In [8]:
import pandas as pd
df=pd.read_csv("sales.csv")
print("shape:",df.shape)

print("Data type:",df.dtypes)

print("Missing value per col:")
print(df.isnull().sum())

#Handling missing value..
df=df.dropna(subset=["customer_name"])

df["quantity"]=df["quantity"].fillna(1)

print("After handaling null value:",df.shape)
#Bad date fix
df["order_date"]=pd.to_datetime(df["order_date"],errors="coerce")
df=df.dropna(subset=["order_date"])

print("After fixing the date:",df.shape)
df['revenue']=df["quantity"]*df["unit_price"]
# Month column for reporting later
df['order_month'] = df['order_date'].dt.to_period('M').astype(str)

# Standardise text
df['region']  = df['region'].str.strip().str.title()
df['product'] = df['product'].str.strip().str.title()

# Validate — make sure nothing slipped through
assert df['order_id'].isnull().sum()   == 0, "Nulls in order_id!"
assert df['revenue'].isnull().sum()    == 0, "Nulls in revenue!"
assert df['order_date'].isnull().sum() == 0, "Nulls in order_date!"

print("All checks passed!")
print("Final clean data shape:", df.shape)
print(df.head())

from sqlalchemy import create_engine, text

# Connect to MySQL
engine = create_engine(
    "mysql+pymysql://root:@localhost:3306/sales_db"
)

# Test connection
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
    print("Connected to MySQL successfully!")

    # Load clean data into MySQL
df.to_sql(
    name="sales",
    con=engine,
    if_exists="replace",
    index=False
)

print("Data loaded successfully!")

shape: (60, 9)
Data type: order_id           int64
order_date        object
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
region            object
salesperson       object
dtype: object
Missing value per col:
order_id         0
order_date       0
customer_name    3
product          0
category         0
quantity         8
unit_price       0
region           0
salesperson      0
dtype: int64
After handaling null value: (57, 9)
After fixing the date: (54, 9)
All checks passed!
Final clean data shape: (54, 11)
   order_id order_date customer_name   product     category  quantity  \
0      1001 2024-01-03   Riya Sharma    Laptop  Electronics       2.0   
1      1002 2024-01-05    Amit Patel     Phone  Electronics       1.0   
3      1004 2024-01-10    Priya Nair  Keyboard  Accessories       1.0   
4      1005 2024-01-12   Rahul Desai    Laptop  Electronics       1.0   
6      1007 2024-01-15   Karan Singh   Monit

In [ ]:
import sys
!{sys.executable} -m pip install pymysql